# Frame sampling calculations

In [24]:
import ipywidgets as widgets
from IPython.display import HTML, display


In [25]:
def update_buffer_values(b, fps, n):
    buffer_duration = b / fps
    spacing = buffer_duration / (n - 1) if n > 1 else 0

    html = f"""
    <div style="font-family: monospace; padding: 15px; background: #f0f0f0; border-radius: 5px;">
        <h3>Inference Parameters</h3>
        <p><b>Buffer Duration:</b> {buffer_duration:.2f} seconds</p>
        <p><b>Spacing Between Frames:</b> {spacing:.3f} seconds</p>
        <p><b>Visual Tokens (8 frames):</b> ~{int(1426 * 8)} tokens</p>
    </div>
    """
    return html


# Sliders
b_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description="Buffer Size (frames):",
    style={"description_width": "200px"},
)
fps_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    step=1,
    description="FPS:",
    style={"description_width": "200px"},
)
n_slider = widgets.IntSlider(
    value=8,
    min=2,
    max=16,
    step=1,
    description="Frames to Sample:",
    style={"description_width": "200px"},
)

output = widgets.Output()


def on_slider_change(change):
    with output:
        output.clear_output()
        display(
            HTML(update_buffer_values(b_slider.value, fps_slider.value, n_slider.value))
        )


b_slider.observe(on_slider_change, names="value")
fps_slider.observe(on_slider_change, names="value")
n_slider.observe(on_slider_change, names="value")

# Display
display(widgets.VBox([b_slider, fps_slider, n_slider, output]))
on_slider_change(None)  # Initial display


In [ ]:
def update_resolution(aspect_ratio, input_p, output_p):
    # Common aspect ratios
    ratios = {
        "16:9": 16 / 9,
        "4:3": 4 / 3,
        "1:1": 1.0,
    }
    ratio = ratios.get(aspect_ratio, 16 / 9)

    # Resolution presets
    presets = {
        "720p": (1280, 720),
        "1080p": (1920, 1080),
        "1440p": (2560, 1440),
        "2160p": (3840, 2160),
    }

    start_width, start_height = presets[input_p]
    # Adjust height based on aspect ratio
    start_height = int(start_width / ratio)

    # Output resolution
    output_presets = {
        "480p": (854, 480),
        "640p": (1137, 640),
        "720p": (1280, 720),
        "1080p": (1920, 1080),
    }
    end_width, end_height = output_presets[output_p]
    end_height = int(end_width / ratio)

    # Round to multiple of 28 (Qwen2.5VL requirement)
    end_height = (end_height // 28) * 28
    end_width = (end_width // 28) * 28

    # Estimate tokens
    patches_width = end_width // 14
    patches_height = end_height // 14
    tokens_per_frame = patches_width * patches_height
    total_tokens = tokens_per_frame * 8

    # VRAM scales with token count
    base_activation_vram = 3.0
    activation_vram = base_activation_vram * (tokens_per_frame / 1426)
    estimated_vram = 2.0 + activation_vram + 1.0

    html = f"""
    <div style="font-family: monospace; padding: 15px; background: #e8f4f8; border-radius: 5px;">
        <h3>Resolution</h3>
        <p><b>Input Resolution ({input_p}):</b> {start_width}×{start_height}</p>
        <p><b>Downscaled Resolution ({output_p}):</b> {end_width}×{end_height}</p>
        <p><b>Aspect Ratio:</b> {aspect_ratio}</p>
        <p><b>Patches per frame:</b> {patches_width}×{patches_height} = {tokens_per_frame} tokens</p>
        <p><b>Total tokens (8 frames):</b> {total_tokens}</p>
        <p><b>VRAM estimate (batch size 1):</b> ~{estimated_vram:.1f} GB</p>
        <p style="font-size: 0.9em; color: #555;"><i>Safe on V100 32GB up to ~8 GB.</i></p>
    </div>
    """
    return html


# Dropdowns
ratio_dropdown = widgets.Dropdown(
    options=["16:9", "4:3", "1:1"],
    value="16:9",
    description="Aspect Ratio:",
    style={"description_width": "200px"},
)

input_p_dropdown = widgets.Dropdown(
    options=["720p", "1080p", "1440p", "2160p"],
    value="1080p",
    description="Input Resolution:",
    style={"description_width": "200px"},
)

output_p_dropdown = widgets.Dropdown(
    options=["480p", "640p", "720p", "1080p"],
    value="640p",
    description="Target Resolution:",
    style={"description_width": "200px"},
)

res_output = widgets.Output()


def on_res_change(change):
    with res_output:
        res_output.clear_output()
        display(
            HTML(
                update_resolution(
                    ratio_dropdown.value,
                    input_p_dropdown.value,
                    output_p_dropdown.value,
                )
            )
        )


ratio_dropdown.observe(on_res_change, names="value")
input_p_dropdown.observe(on_res_change, names="value")
output_p_dropdown.observe(on_res_change, names="value")

# Style
for dropdown in [ratio_dropdown, input_p_dropdown, output_p_dropdown]:
    dropdown.layout = widgets.Layout(width="800px")

display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Resolution</h3>"),
            ratio_dropdown,
            input_p_dropdown,
            output_p_dropdown,
            res_output,
        ],
        layout=widgets.Layout(width="900px"),
    )
)

on_res_change(None)  # Initial display


In [27]:
# All together with styled sliders
all_sliders = widgets.VBox(
    [
        widgets.HTML("<h2>📊 Inference Config Calculator</h2>"),
        widgets.HTML("<h3>Buffer & Frame Spacing</h3>"),
        b_slider,
        fps_slider,
        n_slider,
        output,
        widgets.HTML("<h3>Resolution</h3>"),
        ratio_dropdown,
        input_p_dropdown,
        output_p_dropdown,
        res_output,
    ],
    layout=widgets.Layout(width="900px"),
)

# Style the sliders
for slider in [b_slider, fps_slider, n_slider]:
    slider.layout = widgets.Layout(width="800px")
    slider.style = widgets.SliderStyle(
        handle_color="#2c3e50", description_width="200px"
    )

# Style the dropdowns
for dropdown in [ratio_dropdown, input_p_dropdown, output_p_dropdown]:
    dropdown.layout = widgets.Layout(width="800px")

display(all_sliders)
